# Embeddings in Practice with the Gemini API

This notebook goes with the article [Embeddings and Dense Vector Search: A Practical Primer](article.md). You'll do five things:

1. Implement **cosine similarity** and check it on toy vectors.
2. Turn sentences into **embedding vectors** with Google's `gemini-embedding-001`.
3. Compare sentences, and design a **fair experiment** that changes one thing at a time.
4. Test the famous **king − man + woman ≈ queen** analogy against a baseline.
5. Build a **tiny semantic search engine** in a dozen lines.

Everything runs on the **free tier** of the Gemini API.

### Setup

1. Get a free API key at <https://aistudio.google.com/apikey>.
2. Copy `.env.example` (in the repository root) to `.env` and paste your key there. If you skip this step, the next cell asks for the key.
3. Install the dependencies: `pip install -r requirements.txt`.

In [1]:
import os
from getpass import getpass

import numpy as np
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()  # reads GEMINI_API_KEY from a .env file in this folder or a parent folder
if not os.getenv("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass("Gemini API key: ")

client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"],
    # retry automatically if the free tier rate limit (HTTP 429) is hit
    http_options=types.HttpOptions(retry_options=types.HttpRetryOptions(attempts=5, initial_delay=2.0)),
)
EMBEDDING_MODEL = os.getenv("GEMINI_EMBEDDING_MODEL", "gemini-embedding-001")
print("Using", EMBEDDING_MODEL)

Using gemini-embedding-001


## 1. Cosine similarity

Two embeddings are "close" when they point in a similar **direction**. Cosine similarity measures exactly that. It is the cosine of the angle between two vectors:

$$\cos\theta = \frac{\mathbf{a}\cdot\mathbf{b}}{\lVert\mathbf{a}\rVert\,\lVert\mathbf{b}\rVert}$$

The score runs from **1** (same direction) through **0** (unrelated, at right angles) to **−1** (opposite directions). Because we divide by the lengths, only the direction counts: a vector and a stretched copy of it score a perfect 1.

In [2]:
def cosine_similarity(a, b) -> float:
    """Cosine of the angle between two vectors: 1 = same direction, 0 = unrelated, -1 = opposite."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))


a = [1, 2, 3]
print("same direction, twice as long:", round(cosine_similarity(a, [2, 4, 6]), 4))
print("at right angles:              ", round(cosine_similarity(a, [3, 0, -1]), 4))
print("opposite direction:           ", round(cosine_similarity(a, [-1, -2, -3]), 4))
print("partly aligned:               ", round(cosine_similarity(a, [3, 2, 1]), 4))

same direction, twice as long: 1.0
at right angles:               0.0
opposite direction:            -1.0
partly aligned:                0.7143


## 2. Turning text into vectors

`embed_content` accepts a **list** of texts and returns one vector per text, so a whole batch costs a single request. This matters on the free tier, which limits requests per minute.

`task_type` tells the model what the vectors are for. Use `SEMANTIC_SIMILARITY` to compare texts with each other. For search, use `RETRIEVAL_DOCUMENT` for stored texts and `RETRIEVAL_QUERY` for questions.

In [3]:
def embed(texts, task_type="SEMANTIC_SIMILARITY") -> np.ndarray:
    """Return one embedding per text, as an array of shape (len(texts), dimensions)."""
    response = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=list(texts),
        config=types.EmbedContentConfig(task_type=task_type),
    )
    return np.array([e.values for e in response.embeddings])


vector = embed(["I love puppies!"])[0]
print("dimensions:  ", vector.shape[0])
print("first values:", np.round(vector[:6], 4))
print("length (L2): ", round(float(np.linalg.norm(vector)), 4))

dimensions:   3072
first values: [-0.0158 -0.0032  0.0241 -0.0577 -0.0226  0.0227]
length (L2):  1.0


An embedding is a long list of real numbers. No single number means anything on its own; the meaning lives in the pattern across all of them. At its full size, `gemini-embedding-001` returns vectors of **length 1**, so for these vectors the cosine similarity equals the plain dot product.

## 3. Comparing sentences

We embed every sentence in **one request**, then compare pairs. The first two comparisons repeat the classic demo. The last three form a **controlled experiment**. *"I love puppies!"* vs. *"I dislike cats!"* changes two things at once, the feeling and the animal, so on its own it can't tell us which change lowered the score. Changing one thing at a time can.

In [4]:
sentences = [
    "I love puppies!",       # 0  reference
    "I love dogs!",          # 1  similar meaning
    "I dislike cats!",       # 2  different feeling AND different animal
    "I love cats!",          # 3  only the animal changes
    "I dislike puppies!",    # 4  only the feeling changes
    "The quarterly tax report is due on Friday.",  # 5  unrelated topic
]
vectors = embed(sentences)

pairs = [(0, 1), (0, 2), (0, 3), (0, 4), (0, 5)]
for i, j in pairs:
    score = cosine_similarity(vectors[i], vectors[j])
    print(f"{score:.3f}   {sentences[i]!r}  vs  {sentences[j]!r}")

0.965   'I love puppies!'  vs  'I love dogs!'
0.769   'I love puppies!'  vs  'I dislike cats!'
0.900   'I love puppies!'  vs  'I love cats!'
0.848   'I love puppies!'  vs  'I dislike puppies!'
0.662   'I love puppies!'  vs  'The quarterly tax report is due on Friday.'


**How to read these numbers:**

- Scores are **relative**. Each model has its own range, so there is no universal "similar enough" threshold. Even the unrelated sentence scores well above 0.
- *"I dislike puppies!"* still scores **high** against *"I love puppies!"*. Embeddings capture what a sentence is *about* (puppies, feelings) more strongly than its polarity. Opposite sentiment is **not** the same as an opposite vector.
- Compare the two single-change pairs to see which change mattered more for this model.

## 4. Vector arithmetic: king − man + woman ≈ queen

If the step from *man* to *woman* is roughly a fixed direction in embedding space, adding it to *king* should land near *queen*. A high score alone proves little, though, because *king* and *queen* are already related. A fair test needs two things:

1. **A baseline.** Is the result closer to *queen* than *king* itself was?
2. **A ranking.** Is *queen* the nearest candidate once the input words are left out?

In [5]:
words = ["king", "man", "woman", "queen", "princess", "prince", "monarch", "daughter", "banana"]
word_vectors = dict(zip(words, embed(words)))

result = word_vectors["king"] - word_vectors["man"] + word_vectors["woman"]

print(f"result vs queen : {cosine_similarity(result, word_vectors['queen']):.3f}")
print(f"king   vs queen : {cosine_similarity(word_vectors['king'], word_vectors['queen']):.3f}   <- baseline")

print("\nNearest candidates (input words excluded):")
candidates = [w for w in words if w not in ("king", "man", "woman")]
ranking = sorted(candidates, key=lambda w: cosine_similarity(result, word_vectors[w]), reverse=True)
for w in ranking:
    print(f"  {cosine_similarity(result, word_vectors[w]):.3f}  {w}")

result vs queen : 0.891
king   vs queen : 0.908   <- baseline

Nearest candidates (input words excluded):
  0.891  queen
  0.850  princess
  0.844  monarch
  0.794  daughter
  0.792  prince
  0.752  banana


**What our run showed:** *queen* was the nearest candidate (0.891), ahead of *princess* (0.850), so the **ranking test passes**. But the **baseline test fails**: *king* on its own was already slightly closer to *queen* (0.908) than the arithmetic result was.

So with this sentence-level model, the arithmetic keeps us in the "royalty" neighbourhood and points toward *queen*, but it does **not** make the vector more queen-like than *king* already was. The famous demo comes from word-level models such as word2vec, and it is weaker in modern embedding models. Always check claims like this against a baseline.

## 5. A tiny semantic search engine

This is the core of **dense vector search**, and of the "retrieval" in RAG:

1. Embed the documents once (`RETRIEVAL_DOCUMENT`).
2. Embed the question (`RETRIEVAL_QUERY`).
3. Rank the documents by cosine similarity.

The query below shares almost no keywords with the best answer ("feed" vs. "meals", "young dog" vs. "puppies"). Embeddings bridge that gap; plain keyword search would not.

In [6]:
documents = [
    "Puppies need several small meals a day while they grow.",
    "Adult dogs are usually fed once or twice daily.",
    "Kittens sleep up to 20 hours a day.",
    "A carburetor mixes air and fuel for an internal combustion engine.",
    "The queen is the most powerful piece in chess.",
]
doc_vectors = embed(documents, task_type="RETRIEVAL_DOCUMENT")


def search(query: str, k: int = 3):
    query_vector = embed([query], task_type="RETRIEVAL_QUERY")[0]
    scores = [cosine_similarity(query_vector, d) for d in doc_vectors]
    best = np.argsort(scores)[::-1][:k]
    return [(scores[i], documents[i]) for i in best]


for score, doc in search("How often should I feed my young dog?"):
    print(f"{score:.3f}  {doc}")

0.757  Puppies need several small meals a day while they grow.
0.735  Adult dogs are usually fed once or twice daily.
0.581  Kittens sleep up to 20 hours a day.


## 6. Smaller vectors

`gemini-embedding-001` is trained so that the **first** dimensions carry the most information (Matryoshka representation learning). You can request shorter vectors with `output_dimensionality` to save storage and speed up search. Shortened vectors are **not** length 1, so normalize them before using a plain dot product.

In [7]:
short = client.models.embed_content(
    model=EMBEDDING_MODEL,
    contents=["I love puppies!", "I love dogs!"],
    config=types.EmbedContentConfig(task_type="SEMANTIC_SIMILARITY", output_dimensionality=768),
)
short_vectors = np.array([e.values for e in short.embeddings])
print("shape:", short_vectors.shape)
print("length before normalizing:", np.round(np.linalg.norm(short_vectors, axis=1), 4))

short_vectors /= np.linalg.norm(short_vectors, axis=1, keepdims=True)
print("length after normalizing: ", np.round(np.linalg.norm(short_vectors, axis=1), 4))
print(f"similarity at 768 dims:  {short_vectors[0] @ short_vectors[1]:.3f}")
print(f"similarity at full size: {cosine_similarity(vectors[0], vectors[1]):.3f}")

shape: (2, 768)
length before normalizing: [0.5867 0.5876]
length after normalizing:  [1. 1.]
similarity at 768 dims:  0.964
similarity at full size: 0.965


## Recap

- Embeddings turn text into vectors in which **direction encodes meaning**.
- **Cosine similarity** compares directions, and scores are only meaningful **relative to each other**.
- Good experiments change **one variable at a time** and use **baselines**.
- **Semantic search** is just: embed the documents, embed the query, rank by similarity.

Next: [02-RAG](../02-RAG/) puts this search step in front of a language model so it can answer questions about documents it has never seen.